# Rentfaster Ingest — Step 1

Runs `rentfaster_ingest.py` in Colab. Manual monthly workflow: capture `map.json`
payloads from the Rentfaster map (DevTools -> Network tab -> `map.json` -> Response
-> Save Response As...), upload them here, then run the ingest.

No scraping happens in this notebook — it only parses files you provide.

## 1. Mount Google Drive (recommended for persistent storage)

Skip this cell if you'd rather keep everything in the ephemeral `/content` runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/rf_data"  # change if you prefer a different path

## 2. Install/import dependencies

`pandas` ships with Colab. `pyarrow` (for parquet output) is usually preinstalled;
the script falls back to CSV automatically if it isn't available.

In [ ]:
!pip install -q pyarrow

## 3. Get `rentfaster_ingest.py` into the runtime

Two options — use whichever is easier:

**Option A: clone the repo (recommended if this notebook is opened from GitHub)**

In [ ]:
import os, sys

REPO_URL = "https://github.com/neilmah12/rf-scraping-project.git"
REPO_DIR = "/content/rf-scraping-project"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}

sys.path.insert(0, f"{REPO_DIR}/src")

**Option B: upload `rentfaster_ingest.py` directly** (skip Option A if you use this)

In [ ]:
from google.colab import files
uploaded = files.upload()  # select rentfaster_ingest.py
import sys
sys.path.insert(0, "/content")

In [ ]:
from rentfaster_ingest import ingest_snapshot, rent_changes

## 4. Upload this month's map.json capture(s)

Grab 2-4 zoomed-in quadrant payloads (NW/NE/SW/SE of the map) to beat the ~800
listing cap per response. Upload all of them at once below.

In [ ]:
from google.colab import files
uploaded_payloads = files.upload()  # select one or more map.json files
payload_files = list(uploaded_payloads.keys())
payload_files

## 5. Run the ingest

Set `snapshot_date` to today (or whatever date you're capturing for). Re-running
with the same `snapshot_date` replaces that snapshot (idempotent).

In [ ]:
from datetime import date

master = ingest_snapshot(
    payload_files,
    snapshot_date=date.today().isoformat(),
    data_dir=DATA_DIR,
)
master.head()

## 6. Rent-change signal (optional, once you have 2+ snapshots)

In [ ]:
chg = rent_changes(data_dir=DATA_DIR)
chg.head(20)

## 7. Download CSV outputs (optional, for Excel review)

In [ ]:
from google.colab import files as colab_files
colab_files.download(f"{DATA_DIR}/listings_master.csv")